# Notebook 04 — Gate Fidelity Landscape

This notebook turns the Rydberg blockade model into a simple **quantum information benchmark**.

We define a target entangled state and compute its fidelity under coherent and open-system dynamics.

Target Bell state:

$$
|\Psi^+\rangle = \frac{|gr\rangle + |rg\rangle}{\sqrt{2}}
$$

Fidelity with a density matrix $\rho$ is:

$$
F = \langle \Psi^+ | \rho | \Psi^+ \rangle
$$

The goal is to map how fidelity depends on:
- drive strength $\Omega$
- interaction strength $V$
- spontaneous emission $\gamma$
- pure dephasing $\gamma_\phi$


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, mesolve

## Basis states, operators, and target state

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

psi_target = (gr + rg).unit()
proj_target = psi_target * psi_target.dag()

## Hamiltonian and collapse operators

In [ ]:
def two_atom_hamiltonian(omega: float, delta: float, V: float):
    drive = 0.5 * omega * (sx1 + sx2)
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def collapse_operators(gamma_decay: float = 0.0, gamma_phi: float = 0.0):
    c_ops = []
    if gamma_decay > 0:
        c_ops.append(np.sqrt(gamma_decay) * sm1)
        c_ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        c_ops.append(np.sqrt(gamma_phi) * n1)
        c_ops.append(np.sqrt(gamma_phi) * n2)
    return c_ops

## Fidelity helpers

In [ ]:
def fidelity_trace(omega: float, delta: float, V: float,
                   gamma_decay: float = 0.0,
                   gamma_phi: float = 0.0,
                   t_final: float = 8.0,
                   n_steps: int = 300):
    """Return time trace of Bell-state fidelity."""
    H = two_atom_hamiltonian(omega, delta, V)
    c_ops = collapse_operators(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    times = np.linspace(0.0, t_final, n_steps)
    result = mesolve(H, gg, times, c_ops, [proj_target, n1 + n2])
    fidelities = np.real(result.expect[0])
    total_excitation = np.real(result.expect[1])
    return times, fidelities, total_excitation


def max_fidelity(omega: float, delta: float, V: float,
                 gamma_decay: float = 0.0,
                 gamma_phi: float = 0.0,
                 t_final: float = 8.0,
                 n_steps: int = 300):
    _, fidelities, _ = fidelity_trace(
        omega=omega,
        delta=delta,
        V=V,
        gamma_decay=gamma_decay,
        gamma_phi=gamma_phi,
        t_final=t_final,
        n_steps=n_steps,
    )
    return float(np.max(fidelities))

## Time traces: coherent and noisy fidelity

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0

cases = [
    (0.0, 0.0, 'coherent'),
    (0.03, 0.0, 'decay only'),
    (0.0, 0.06, 'dephasing only'),
    (0.03, 0.06, 'decay + dephasing'),
]

plt.figure(figsize=(8, 5))
for gamma_decay, gamma_phi, label in cases:
    times, fidelities, _ = fidelity_trace(
        omega=omega,
        delta=delta,
        V=V,
        gamma_decay=gamma_decay,
        gamma_phi=gamma_phi,
        t_final=10.0,
        n_steps=400,
    )
    plt.plot(times, fidelities, label=label)

plt.xlabel('Time')
plt.ylabel('Bell-state fidelity')
plt.title('Entanglement fidelity under coherent and open-system dynamics')
plt.legend()
plt.tight_layout()
plt.show()

## Coherent fidelity landscape over $(\Omega, V)$

In [ ]:
omega_vals = np.linspace(0.2, 3.5, 45)
V_vals = np.linspace(0.0, 8.0, 45)

fidelity_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V in enumerate(V_vals):
    for j, omega in enumerate(omega_vals):
        fidelity_landscape[i, j] = max_fidelity(
            omega=omega,
            delta=0.0,
            V=V,
            gamma_decay=0.0,
            gamma_phi=0.0,
            t_final=8.0,
            n_steps=220,
        )

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    fidelity_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, fidelity_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Max Bell-state fidelity over $(\Omega, V)$')
plt.colorbar(im, label='max fidelity')
plt.tight_layout()
plt.show()

## Noise-dependent fidelity landscape over $(\gamma, \gamma_\phi)$

In [ ]:
gamma_decay_grid = np.linspace(0.0, 0.20, 36)
gamma_phi_grid = np.linspace(0.0, 0.20, 36)

noise_fidelity = np.zeros((len(gamma_phi_grid), len(gamma_decay_grid)))

for i, gamma_phi in enumerate(gamma_phi_grid):
    for j, gamma_decay in enumerate(gamma_decay_grid):
        noise_fidelity[i, j] = max_fidelity(
            omega=1.0,
            delta=0.0,
            V=4.0,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
            t_final=8.0,
            n_steps=220,
        )

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    noise_fidelity,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_grid[0], gamma_decay_grid[-1], gamma_phi_grid[0], gamma_phi_grid[-1]],
)
plt.contour(gamma_decay_grid, gamma_phi_grid, noise_fidelity, colors='white', linewidths=0.4)
plt.xlabel(r'Spontaneous emission $\gamma$')
plt.ylabel(r'Pure dephasing $\gamma_\phi$')
plt.title(r'Max Bell-state fidelity over $(\gamma, \gamma_\phi)$')
plt.colorbar(im, label='max fidelity')
plt.tight_layout()
plt.show()

## Best point in the coherent $(\Omega, V)$ scan

In [ ]:
best_index = np.unravel_index(np.argmax(fidelity_landscape), fidelity_landscape.shape)
best_V = V_vals[best_index[0]]
best_omega = omega_vals[best_index[1]]
best_F = fidelity_landscape[best_index]

print(f'Best coherent scan point: Omega = {best_omega:.3f}, V = {best_V:.3f}, max fidelity = {best_F:.4f}')

## Interpretation

This notebook converts the Rydberg model into a simple entanglement benchmark:

- the coherent $(\Omega, V)$ landscape identifies regions where blockade-assisted dynamics best approach the Bell target,
- the noise landscape shows how spontaneous emission and dephasing reduce achievable fidelity,
- the time traces make it easy to see whether fidelity peaks are sharp, broad, or fragile.

This is a useful bridge from AMO modeling to quantum information performance metrics.


## Optional next steps

Natural extensions from here are:

- optimize pulse duration rather than taking the maximum over a fixed window,
- add shaped pulses $\Omega(t)$,
- evaluate a CZ-like truth table instead of a Bell target,
- include a distance-dependent interaction model $V(R)$,
- move from landscape scans to parameter optimization.
